# Track C — O/D mathematics (exact; runs on CPU; independent of any training)
Original O/D (Phase-1 definitions), the repaired O/D of the project note (independent implementation), T4 interaction-order bound, and the predeclared C1–C5 classification. No neural O/D experiment is run unless a distinctive prediction exists.

In [ ]:
# ===================== CONFIGURATION (the only cell you need to edit) =====================
import os
PRESET = os.environ.get("COR_NB_PRESET", "pilot")    # "pilot" (technical validation) | "standard" | "extended" | "high"   ("smoke" = CPU software test)
USE_DRIVE = os.environ.get("COR_NB_DRIVE", "1") == "1"   # keep every checkpoint, registry record and result on Google Drive (survives disconnects)
DRIVE_DIR = "/content/drive/MyDrive/COR_XX_RUNS"
PACKAGE_ZIP = "/content/drive/MyDrive/cor_inv20_colab_three_track_research.zip"   # or upload the ZIP to /content
BUDGET_HOURS = None       # stop launching new jobs after this many hours (running jobs finish; nothing is shortened)
CONCURRENCY = "auto"      # "auto" = measured by the benchmark; or an integer
RETRY_FAILED = False      # True: failed jobs get a new attempt (their failed attempts stay in the registry)
DEVICE = os.environ.get("COR_NB_DEVICE", "auto")   # "auto" | "gpu" | "cpu"
ALLOW_IDENTITY_CHANGE = False   # True only if you deliberately continue a run root on a different GPU type / software stack (recorded)
RUN_ROOT = os.environ.get("COR_NB_RUN_ROOT") or None   # None = DRIVE_DIR/<preset> (Drive) or /content/COR_XX_RUNS/<preset>
# (the COR_NB_* environment overrides exist only for the package's automated notebook tests)

In [ ]:
# Locate / extract the package and make it importable (idempotent)
import os, sys, zipfile, glob
CANDIDATES = ["/content/COR_Investigation_XX", os.path.abspath("COR_Investigation_XX"), os.path.abspath("..")]
PKG = next((c for c in CANDIDATES if os.path.exists(os.path.join(c, "cor_research", "pipeline.py"))), None)
if PKG is None:
    if USE_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive; drive.mount("/content/drive")
    zips = [PACKAGE_ZIP] + glob.glob("/content/cor_inv20_colab_three_track_research*.zip")
    z = next((x for x in zips if os.path.exists(x)), None)
    assert z, "package ZIP not found: upload cor_inv20_colab_three_track_research.zip to /content or set PACKAGE_ZIP"
    with zipfile.ZipFile(z) as zf: zf.extractall("/content")
    PKG = "/content/COR_Investigation_XX"
sys.path.insert(0, PKG); os.chdir(PKG); print("package:", PKG)

In [ ]:
# 2. Exact dependencies (pinned in requirements.txt / requirements-gpu.txt).  If anything had to be installed, the runtime restarts
#    once automatically; then run all cells again from the top (every step is idempotent and resumes).
import importlib.metadata as IM, subprocess, sys, os, shutil
def pins(f): return dict(l.split("#")[0].strip().split("==") for l in open(f) if "==" in l.split("#")[0])
want = pins("requirements.txt"); gpu = shutil.which("nvidia-smi") is not None and DEVICE != "cpu"
if gpu: want.update(pins("requirements-gpu.txt"))
def have(n):
    try: return IM.version(n.split("[")[0])
    except Exception: return None
missing = {n: v for n, v in want.items() if have(n) != v}
if missing:
    print("installing", missing)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"] + (["-r", "requirements-gpu.txt"] if gpu else []), capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-3000:])
    if r.returncode != 0: raise SystemExit("dependency installation FAILED (see the output above); nothing else will run")
    print("dependencies installed; restarting the runtime: run all cells again"); os.kill(os.getpid(), 9)
else: print("all pinned dependencies present:", want)

In [ ]:
# 3. Persistent storage + device (run root on Drive: checkpoints, registry, results)
from cor_research import pipeline as PL
INFO = PL.setup(PRESET, run_root=RUN_ROOT, use_drive=USE_DRIVE, drive_dir=DRIVE_DIR, device='cpu' if DEVICE == 'auto' else DEVICE, allow_identity_change=ALLOW_IDENTITY_CHANGE); INFO

In [ ]:
TRACK_C = PL.track_c()

In [ ]:
import json
for k, v in TRACK_C['repaired_gate'].items(): print(k, {x: v[x] for x in ('R1_max_err_signed_identity', 'M1_O_is_function_of_sorted_posterior', 'R3_gate_flip_following_repaired', 'LP_max_margin_by_order')})
print(json.dumps(TRACK_C['t4_order']['statement']))